# BirdCLEF+ 2026 ONNX Perch + SED Temporal Residual Blend

A larger temporal modeling experiment built on the 0.893 soundscape-calibrated baseline. The notebook keeps the stable ONNX SED + ONNX Perch inference path, learns per-class soundscape blend weights, then trains a small temporal residual model from labeled train-soundscape windows. The residual sees neighboring 5-second windows and Perch embeddings, then contributes a light probability blend at submission time.


## 1. Setup And Configuration

Use fixed Kaggle input roots and attached wheel/model datasets. The default path reproduces the soundscape-calibrated champion before adding a controlled temporal residual stage.


In [1]:
from __future__ import annotations

from pathlib import Path
import copy
import subprocess
import sys
import time
from types import ModuleType
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


class CFG:
    """Configuration for ONNX SED plus ONNX Perch blending."""

    seed = 42
    data_root = Path("/kaggle/input/competitions/birdclef-2026")
    sed_model_dir = Path(
        "/kaggle/input/datasets/tuckerarrants/bc2026-distilled-sed-public"
    )
    perch_onnx_root = Path(
        "/kaggle/input/datasets/tuckerarrants/perch-v2-no-dft-onnx"
    )
    perch_tf_model_dir = Path(
        "/kaggle/input/models/google/"
        "bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1"
    )
    output_path = Path("/kaggle/working/submission.csv")
    calibration_output_path = Path(
        "/kaggle/working/soundscape_blend_calibration.csv"
    )
    temporal_history_path = Path(
        "/kaggle/working/temporal_residual_history.csv"
    )

    sample_rate = 32_000
    clip_seconds = 5
    soundscape_seconds = 60
    n_windows = soundscape_seconds // clip_seconds
    samples_per_window = sample_rate * clip_seconds
    samples_per_file = sample_rate * soundscape_seconds

    sed_n_mels = 256
    sed_n_fft = 2048
    sed_hop_length = 512
    sed_fmin = 20
    sed_fmax = 16_000
    sed_top_db = 80
    sed_smooth_sigma = 0.75
    sed_fold_pattern = "sed_fold*.onnx"

    perch_model_patterns = ("perch_v2_no_dft*.onnx", "perch_v2*.onnx")
    onnx_wheel_pattern = "onnxruntime-*.whl"
    onnx_threads = 4
    batch_files = 4
    perch_weight = 0.15
    proxy_weight = 0.05
    calibration_grid = np.array(
        [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30], dtype=np.float32
    )
    min_positive_for_calibration = 2
    max_calibration_files = None

    temporal_residual_weight = 0.10
    temporal_epochs = 40
    temporal_patience = 6
    temporal_batch_size = 128
    temporal_learning_rate = 1e-3
    temporal_weight_decay = 1e-4
    temporal_validation_fraction = 0.20
    temporal_embedding_hidden = 128
    temporal_hidden = 256
    temporal_dropout = 0.20
    max_pos_weight = 20.0
    torch_threads = 2


np.random.seed(CFG.seed)
print("Competition root:", CFG.data_root)
print("SED model root:", CFG.sed_model_dir)
print("Perch ONNX root:", CFG.perch_onnx_root)


Competition root: /kaggle/input/competitions/birdclef-2026
SED model root: /kaggle/input/datasets/tuckerarrants/bc2026-distilled-sed-public
Perch ONNX root: /kaggle/input/datasets/tuckerarrants/perch-v2-no-dft-onnx


In [2]:
def install_onnxruntime_if_needed() -> ModuleType:
    """Import ONNX Runtime, installing an attached wheel if needed.

    Returns:
        ModuleType: Imported `onnxruntime` module.

    Raises:
        FileNotFoundError: If ONNX Runtime is unavailable and no offline wheel
            is attached under `/kaggle/input`.
    """
    try:
        import onnxruntime as ort

        return ort
    except ImportError:
        wheels = sorted(CFG.perch_onnx_root.glob(CFG.onnx_wheel_pattern))
        if not wheels:
            wheels = sorted(Path("/kaggle/input").glob("**/onnxruntime-*.whl"))
        if not wheels:
            raise FileNotFoundError(
                "No offline onnxruntime wheel found under /kaggle/input."
            )

        wheel = wheels[0]
        print("Installing ONNX Runtime from", wheel)
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-deps",
                str(wheel),
            ],
            check=True,
        )

        import onnxruntime as ort

        return ort


ort = install_onnxruntime_if_needed()
import librosa
import soundfile as sf
from scipy.ndimage import gaussian_filter1d
import torch
from torch import nn

torch.manual_seed(CFG.seed)
torch.set_num_threads(CFG.torch_threads)

print("ONNX Runtime:", ort.__version__)


Installing ONNX Runtime from /kaggle/input/datasets/tuckerarrants/perch-v2-no-dft-onnx/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
ONNX Runtime: 1.24.4


## 2. Submission Contract And Label Mapping

The SED model already predicts the 234 submission columns. ONNX Perch predicts 14,795 global labels, so this notebook uses exact scientific-name matching before any blend is attempted.

In [3]:
sample_submission = pd.read_csv(CFG.data_root / "sample_submission.csv")
taxonomy = pd.read_csv(CFG.data_root / "taxonomy.csv")
target_columns = [c for c in sample_submission.columns if c != "row_id"]
num_classes = len(target_columns)
label_to_pos = {label: idx for idx, label in enumerate(target_columns)}

print("Sample submission:", sample_submission.shape)
print("Taxonomy:", taxonomy.shape)
print("Target columns:", num_classes)


def find_perch_labels_path() -> Path:
    """Find Perch `labels.csv` metadata from attached Kaggle inputs.

    Returns:
        Path: CSV containing Perch label metadata.

    Raises:
        FileNotFoundError: If no compatible Perch label metadata is found.
    """
    preferred = CFG.perch_tf_model_dir / "assets" / "labels.csv"
    if preferred.exists():
        return preferred

    for path in sorted(Path("/kaggle/input").glob("**/labels.csv")):
        try:
            columns = set(pd.read_csv(path, nrows=0).columns)
        except Exception:
            continue
        if {"inat2024_fsd50k", "scientific_name", "label", "name"} & columns:
            return path

    raise FileNotFoundError(
        "Perch labels.csv not found. Attach the Google Perch model or another "
        "input containing compatible Perch label metadata."
    )


def load_perch_labels(path: Path) -> pd.DataFrame:
    """Load Perch labels and normalize the scientific-name column.

    Args:
        path (Path): Perch label metadata CSV.

    Returns:
        pd.DataFrame: Perch labels with `bc_index` and `scientific_name`.
    """
    labels = (
        pd.read_csv(path)
        .reset_index()
        .rename(
            columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"}
        )
    )
    if "scientific_name" not in labels.columns:
        for column in ["label", "labels", "name"]:
            if column in labels.columns:
                labels = labels.rename(columns={column: "scientific_name"})
                break
    if "scientific_name" not in labels.columns:
        raise ValueError(f"No scientific-name column found in {path}")
    labels["scientific_name"] = labels["scientific_name"].astype(str)
    return labels[["bc_index", "scientific_name"]].copy()


def build_exact_perch_mapping(perch_labels: pd.DataFrame) -> pd.DataFrame:
    """Map BirdCLEF submission labels to exact Perch label indices.

    Args:
        perch_labels (pd.DataFrame): Normalized Perch label metadata.

    Returns:
        pd.DataFrame: Mapping rows for all submission labels.
    """
    mapping = taxonomy[["primary_label", "scientific_name"]].merge(
        perch_labels, on="scientific_name", how="left"
    )
    mapping = mapping[mapping["primary_label"].isin(target_columns)].copy()
    mapping["target_pos"] = mapping["primary_label"].map(label_to_pos)
    mapping["is_mapped"] = mapping["bc_index"].notna()
    mapping["bc_index"] = mapping["bc_index"].fillna(-1).astype(int)
    mapping["target_pos"] = mapping["target_pos"].astype(int)
    return mapping.sort_values("target_pos").reset_index(drop=True)


def build_named_genus_proxy_map(
    mapping: pd.DataFrame, perch_labels: pd.DataFrame
) -> dict[int, np.ndarray]:
    """Build same-genus proxy candidates for named unmapped targets only.

    Anonymous insect sonotypes are intentionally skipped because their labels do
    not contain a real genus/species name.

    Args:
        mapping (pd.DataFrame): Exact mapping table for submission columns.
        perch_labels (pd.DataFrame): Normalized Perch label metadata.

    Returns:
        dict[int, np.ndarray]: Target-column position to Perch label indices.
    """
    proxy_map = {}
    unmapped = mapping[~mapping["is_mapped"]].copy()
    perch_names = perch_labels["scientific_name"].astype(str)

    for _, row in unmapped.iterrows():
        scientific_name = str(row["scientific_name"])
        if scientific_name.startswith("Insect son"):
            continue
        parts = scientific_name.split()
        if len(parts) < 2:
            continue

        genus = parts[0]
        hits = perch_labels[perch_names.str.match(rf"^{genus}\s", na=False)]
        if len(hits) == 0:
            continue
        proxy_map[int(row["target_pos"])] = hits["bc_index"].to_numpy(
            dtype=np.int32
        )
    return proxy_map


perch_labels_path = find_perch_labels_path()
perch_labels = load_perch_labels(perch_labels_path)
perch_mapping = build_exact_perch_mapping(perch_labels)
proxy_map = build_named_genus_proxy_map(perch_mapping, perch_labels)

mapped = perch_mapping[perch_mapping["is_mapped"]]
unmapped = perch_mapping[~perch_mapping["is_mapped"]]
sonotype_unmapped = unmapped[
    unmapped["scientific_name"].astype(str).str.startswith("Insect son")
]

mapped_target_pos = mapped["target_pos"].to_numpy(dtype=np.int32)
mapped_perch_idx = mapped["bc_index"].to_numpy(dtype=np.int32)
proxy_target_pos = np.array(sorted(proxy_map), dtype=np.int32)

print("Perch labels:", perch_labels.shape, perch_labels_path)
print(f"Exact Perch mapping: {len(mapped)} / {num_classes}")
print(f"Unmapped target columns: {len(unmapped)}")
print(f"Skipped insect sonotypes: {len(sonotype_unmapped)}")
print(f"Named genus proxies: {len(proxy_map)}")
for target_pos in proxy_target_pos:
    label = target_columns[target_pos]
    n_proxy = len(proxy_map[int(target_pos)])
    print(f" - {label}: {n_proxy} same-genus Perch labels")


Sample submission: (3, 235)
Taxonomy: (234, 5)
Target columns: 234
Perch labels: (14795, 2) /kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1/assets/labels.csv
Exact Perch mapping: 203 / 234
Unmapped target columns: 31
Skipped insect sonotypes: 25
Named genus proxies: 6
 - 116570: 2 same-genus Perch labels
 - 1491113: 9 same-genus Perch labels
 - 1595929: 2 same-genus Perch labels
 - 25073: 6 same-genus Perch labels
 - 516975: 3 same-genus Perch labels
 - 74580: 1 same-genus Perch labels


## 3. Load ONNX Models

Both model families run with `CPUExecutionProvider`. SED has multiple folds; Perch uses one no-DFT ONNX model.

In [4]:
def find_sed_folds() -> list[Path]:
    """Find distilled SED ONNX fold files.

    Returns:
        list[Path]: Sorted SED ONNX fold paths.

    Raises:
        FileNotFoundError: If no SED folds are attached.
    """
    folds = sorted(CFG.sed_model_dir.glob(CFG.sed_fold_pattern))
    if not folds:
        folds = sorted(
            Path("/kaggle/input").glob(f"**/{CFG.sed_fold_pattern}")
        )
    if not folds:
        raise FileNotFoundError("No SED ONNX fold files found.")
    return folds


def find_perch_onnx_model() -> Path:
    """Find the ONNX Perch model file.

    Returns:
        Path: ONNX Perch model path.

    Raises:
        FileNotFoundError: If no matching ONNX Perch model is attached.
    """
    for pattern in CFG.perch_model_patterns:
        candidates = sorted(CFG.perch_onnx_root.glob(pattern))
        if not candidates:
            candidates = sorted(Path("/kaggle/input").glob(f"**/{pattern}"))
        if candidates:
            return candidates[0]
    raise FileNotFoundError("No ONNX Perch model found under /kaggle/input.")


def make_session(path: Path) -> ort.InferenceSession:
    """Create an optimized CPU ONNX Runtime session.

    Args:
        path (Path): ONNX model path.

    Returns:
        ort.InferenceSession: Loaded ONNX Runtime CPU session.
    """
    options = ort.SessionOptions()
    options.intra_op_num_threads = CFG.onnx_threads
    options.inter_op_num_threads = 1
    options.graph_optimization_level = (
        ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    )
    return ort.InferenceSession(
        str(path), sess_options=options, providers=["CPUExecutionProvider"]
    )


sed_folds = find_sed_folds()
sed_sessions = [make_session(path) for path in sed_folds]
sed_input_names = [session.get_inputs()[0].name for session in sed_sessions]

perch_model_path = find_perch_onnx_model()
perch_session = make_session(perch_model_path)
perch_input_name = perch_session.get_inputs()[0].name
perch_output_names = [output.name for output in perch_session.get_outputs()]
perch_output_map = {name: idx for idx, name in enumerate(perch_output_names)}

print("SED folds:", len(sed_sessions))
for path in sed_folds:
    print(" -", path)
print("Perch model:", perch_model_path)
print("Perch outputs:", perch_output_names)


SED folds: 5
 - /kaggle/input/datasets/tuckerarrants/bc2026-distilled-sed-public/sed_fold0.onnx
 - /kaggle/input/datasets/tuckerarrants/bc2026-distilled-sed-public/sed_fold1.onnx
 - /kaggle/input/datasets/tuckerarrants/bc2026-distilled-sed-public/sed_fold2.onnx
 - /kaggle/input/datasets/tuckerarrants/bc2026-distilled-sed-public/sed_fold3.onnx
 - /kaggle/input/datasets/tuckerarrants/bc2026-distilled-sed-public/sed_fold4.onnx
Perch model: /kaggle/input/datasets/tuckerarrants/perch-v2-no-dft-onnx/perch_v2_no_dft.onnx
Perch outputs: ['embedding', 'spatial_embedding', 'spectrogram', 'label']


## 4. Audio And Prediction Helpers

One 60-second file is loaded once, split into 12 windows, and converted into both model inputs: log-mel tensors for SED and raw waveform windows for Perch.

In [5]:
def sigmoid(x: np.ndarray) -> np.ndarray:
    """Apply a numerically stable sigmoid transform.

    Args:
        x (np.ndarray): Logit array.

    Returns:
        np.ndarray: Probability array with the same shape as `x`.
    """
    x = np.clip(x, -40, 40)
    return 1.0 / (1.0 + np.exp(-x))


def logit(probs: np.ndarray, eps: float = 1e-5) -> np.ndarray:
    """Convert probabilities to clipped logits.

    Args:
        probs (np.ndarray): Probability array.
        eps (float): Numerical clipping value.

    Returns:
        np.ndarray: Logit array with the same shape as `probs`.
    """
    probs = np.clip(probs, eps, 1.0 - eps)
    return np.log(probs / (1.0 - probs)).astype(np.float32)


def load_audio_60s(audio_path: Path) -> np.ndarray:
    """Load one soundscape as a fixed-length mono waveform.

    Args:
        audio_path (Path): Soundscape OGG path.

    Returns:
        np.ndarray: Float32 waveform with exactly 60 seconds of audio.
    """
    audio, sr = sf.read(str(audio_path), dtype="float32", always_2d=False)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if sr != CFG.sample_rate:
        audio = librosa.resample(
            audio, orig_sr=sr, target_sr=CFG.sample_rate
        ).astype(np.float32)

    if len(audio) < CFG.samples_per_file:
        audio = np.pad(audio, (0, CFG.samples_per_file - len(audio)))
    else:
        audio = audio[: CFG.samples_per_file]
    return audio.astype(np.float32, copy=False)


def waveform_to_windows(audio: np.ndarray) -> np.ndarray:
    """Reshape one 60-second waveform into 5-second windows.

    Args:
        audio (np.ndarray): Fixed-length soundscape waveform.

    Returns:
        np.ndarray: Array shaped `(12, 160000)`.
    """
    return audio.reshape(CFG.n_windows, CFG.samples_per_window)


def windows_to_sed_logmel(windows: np.ndarray) -> np.ndarray:
    """Convert waveform windows into normalized SED log-mel tensors.

    Args:
        windows (np.ndarray): Raw waveform windows.

    Returns:
        np.ndarray: Float32 tensor shaped `(12, 1, n_mels, frames)`.
    """
    features = []
    for window in windows:
        mel = librosa.feature.melspectrogram(
            y=window,
            sr=CFG.sample_rate,
            n_fft=CFG.sed_n_fft,
            hop_length=CFG.sed_hop_length,
            n_mels=CFG.sed_n_mels,
            fmin=CFG.sed_fmin,
            fmax=CFG.sed_fmax,
            power=2.0,
        )
        mel_db = librosa.power_to_db(mel, ref=np.max, top_db=CFG.sed_top_db)
        mel_db = (mel_db + CFG.sed_top_db) / CFG.sed_top_db
        features.append(mel_db.astype(np.float32))
    return np.expand_dims(np.stack(features, axis=0), axis=1)


def predict_sed(windows: np.ndarray) -> np.ndarray:
    """Predict SED probabilities for one soundscape.

    Args:
        windows (np.ndarray): Raw waveform windows.

    Returns:
        np.ndarray: Probability matrix shaped `(12, 234)`.
    """
    batch = windows_to_sed_logmel(windows)
    logits_sum = np.zeros((CFG.n_windows, num_classes), dtype=np.float32)

    for session, input_name in zip(sed_sessions, sed_input_names):
        outputs = session.run(None, {input_name: batch})
        clip_logits = outputs[0]
        frame_max = outputs[1].max(axis=1)
        logits_sum += 0.5 * clip_logits + 0.5 * frame_max

    logits = logits_sum / len(sed_sessions)
    logits = gaussian_filter1d(
        logits, sigma=CFG.sed_smooth_sigma, axis=0, mode="nearest"
    )
    return sigmoid(logits).astype(np.float32)


def predict_perch_outputs(
    windows: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Predict mapped Perch probabilities and embeddings.

    Args:
        windows (np.ndarray): Raw waveform windows.

    Returns:
        tuple[np.ndarray, np.ndarray]: Mapped probability matrix shaped
            `(12, 234)` and embedding matrix shaped `(12, embedding_dim)`.
    """
    outputs = perch_session.run(None, {perch_input_name: windows})
    if "label" not in perch_output_map:
        raise KeyError(f"Perch output 'label' not found: {perch_output_names}")
    if "embedding" not in perch_output_map:
        raise KeyError(
            f"Perch output 'embedding' not found: {perch_output_names}"
        )

    logits = outputs[perch_output_map["label"]].astype(np.float32)
    embeddings = outputs[perch_output_map["embedding"]].astype(np.float32)
    probs = np.zeros((CFG.n_windows, num_classes), dtype=np.float32)
    probs[:, mapped_target_pos] = sigmoid(logits[:, mapped_perch_idx])

    for target_pos, perch_indices in proxy_map.items():
        proxy_logits = logits[:, perch_indices].max(axis=1)
        probs[:, target_pos] = sigmoid(proxy_logits)
    return probs, embeddings


def predict_perch_mapped(windows: np.ndarray) -> np.ndarray:
    """Predict exact and proxy-mapped Perch probabilities.

    Args:
        windows (np.ndarray): Raw waveform windows.

    Returns:
        np.ndarray: Probability matrix shaped `(12, 234)`.
    """
    perch_probs, _ = predict_perch_outputs(windows)
    return perch_probs


def blend_predictions(
    sed_probs: np.ndarray, perch_probs: np.ndarray
) -> np.ndarray:
    """Blend SED, exact Perch, and narrow proxy Perch probabilities.

    Args:
        sed_probs (np.ndarray): SED probabilities shaped `(12, 234)`.
        perch_probs (np.ndarray): Perch probabilities shaped `(12, 234)`.

    Returns:
        np.ndarray: Blended probabilities shaped `(12, 234)`.
    """
    blended = sed_probs.copy()
    w = CFG.perch_weight
    blended[:, mapped_target_pos] = (1.0 - w) * sed_probs[
        :, mapped_target_pos
    ] + w * perch_probs[:, mapped_target_pos]

    if len(proxy_target_pos) > 0:
        proxy_w = CFG.proxy_weight
        blended[:, proxy_target_pos] = (1.0 - proxy_w) * sed_probs[
            :, proxy_target_pos
        ] + proxy_w * perch_probs[:, proxy_target_pos]

    return np.clip(blended, 0.0, 1.0).astype(np.float32)


def predict_file(audio_path: Path) -> tuple[list[str], np.ndarray]:
    """Predict blended probabilities for one soundscape file.

    Args:
        audio_path (Path): Soundscape OGG path.

    Returns:
        tuple[list[str], np.ndarray]: Row IDs and blended probabilities.
    """
    audio = load_audio_60s(audio_path)
    windows = waveform_to_windows(audio)
    sed_probs = predict_sed(windows)
    perch_probs = predict_perch_mapped(windows)
    blended = blend_predictions(sed_probs, perch_probs)

    stem = audio_path.stem
    row_ids = [
        f"{stem}_{(idx + 1) * CFG.clip_seconds}"
        for idx in range(CFG.n_windows)
    ]
    return row_ids, blended


## 5. Soundscape Calibration And Temporal Residual

First learn the same per-class blend weights used by the 0.893 champion. Then train a compact residual network on labeled train-soundscape windows. The residual uses current, previous, and next calibrated probabilities plus Perch embeddings, so it can learn simple temporal context without the runtime and fragility of a full sequence model.


In [6]:
def parse_seconds(value: object) -> int:
    """Parse a soundscape timestamp into integer seconds.

    Args:
        value (object): Time value from `train_soundscapes_labels.csv`.

    Returns:
        int: Number of seconds.
    """
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        return int(value)
    text = str(value)
    if text.isdigit():
        return int(text)
    return int(pd.to_timedelta(text).total_seconds())


def split_labels(values: pd.Series) -> list[str]:
    """Collect unique labels from a grouped label series.

    Args:
        values (pd.Series): Raw primary-label values.

    Returns:
        list[str]: Sorted unique labels.
    """
    labels = set()
    for value in values:
        if pd.isna(value):
            continue
        for label in str(value).split(";"):
            label = label.strip()
            if label:
                labels.add(label)
    return sorted(labels)


def build_soundscape_targets() -> tuple[pd.DataFrame, np.ndarray]:
    """Build row-level targets from labeled train-soundscape windows.

    Returns:
        tuple[pd.DataFrame, np.ndarray]: Metadata and binary target matrix.
    """
    labels = pd.read_csv(CFG.data_root / "train_soundscapes_labels.csv")
    grouped = (
        labels.groupby(["filename", "start", "end"])["primary_label"]
        .apply(split_labels)
        .reset_index(name="label_list")
    )
    grouped["end_sec"] = grouped["end"].map(parse_seconds)
    grouped["row_id"] = (
        grouped["filename"].str.replace(".ogg", "", regex=False)
        + "_"
        + grouped["end_sec"].astype(str)
    )
    grouped = grouped.sort_values(["filename", "end_sec"]).reset_index(
        drop=True
    )

    targets = np.zeros((len(grouped), num_classes), dtype=np.uint8)
    for row_idx, labels_for_row in enumerate(grouped["label_list"]):
        for label in labels_for_row:
            if label in label_to_pos:
                targets[row_idx, label_to_pos[label]] = 1
    return grouped, targets


def predict_file_components(
    audio_path: Path,
) -> tuple[list[str], np.ndarray, np.ndarray, np.ndarray]:
    """Predict SED, Perch, and embedding components for one soundscape.

    Args:
        audio_path (Path): Soundscape OGG path.

    Returns:
        tuple[list[str], np.ndarray, np.ndarray, np.ndarray]: Row IDs, SED
            probabilities, Perch probabilities, and Perch embeddings.
    """
    audio = load_audio_60s(audio_path)
    windows = waveform_to_windows(audio)
    sed_probs = predict_sed(windows)
    perch_probs, embeddings = predict_perch_outputs(windows)

    stem = audio_path.stem
    row_ids = [
        f"{stem}_{(idx + 1) * CFG.clip_seconds}"
        for idx in range(CFG.n_windows)
    ]
    return row_ids, sed_probs, perch_probs, embeddings


def average_precision_score_binary(
    y_true: np.ndarray, scores: np.ndarray
) -> float:
    """Compute binary average precision without external dependencies.

    Args:
        y_true (np.ndarray): Binary target vector.
        scores (np.ndarray): Prediction scores.

    Returns:
        float: Average precision, or `nan` when no positives exist.
    """
    positives = int(y_true.sum())
    if positives == 0:
        return float("nan")
    order = np.argsort(-scores)
    ranked = y_true[order]
    hit_count = np.cumsum(ranked)
    precision = hit_count / (np.arange(len(ranked)) + 1)
    return float((precision * ranked).sum() / positives)


def macro_average_precision(
    targets: np.ndarray, scores: np.ndarray
) -> float:
    """Compute macro average precision over classes with positives.

    Args:
        targets (np.ndarray): Binary target matrix.
        scores (np.ndarray): Prediction score matrix.

    Returns:
        float: Mean average precision over non-empty classes.
    """
    values = [
        average_precision_score_binary(targets[:, idx], scores[:, idx])
        for idx in range(targets.shape[1])
        if targets[:, idx].sum() > 0
    ]
    return float(np.nanmean(values)) if values else float("nan")


def collect_calibration_predictions(
    metadata: pd.DataFrame,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Score labeled train soundscapes for calibration and residual training.

    Args:
        metadata (pd.DataFrame): Soundscape target metadata.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: SED probabilities, Perch
            probabilities, and Perch embeddings aligned to `metadata` rows.
    """
    filenames = sorted(metadata["filename"].unique())
    if CFG.max_calibration_files is not None:
        filenames = filenames[: CFG.max_calibration_files]

    rows = []
    sed_values = []
    perch_values = []
    embedding_values = []
    start = time.perf_counter()

    for idx, filename in enumerate(filenames, start=1):
        audio_path = CFG.data_root / "train_soundscapes" / filename
        if not audio_path.exists():
            continue
        row_ids, sed_probs, perch_probs, embeddings = predict_file_components(
            audio_path
        )
        rows.extend(row_ids)
        sed_values.append(sed_probs)
        perch_values.append(perch_probs)
        embedding_values.append(embeddings)
        if idx == 1 or idx % 10 == 0 or idx == len(filenames):
            elapsed = time.perf_counter() - start
            print(
                f"Calibration files {idx}/{len(filenames)} in {elapsed:.1f}s"
            )

    sed_df = pd.DataFrame(np.vstack(sed_values), columns=target_columns)
    sed_df.insert(0, "row_id", rows)
    sed_df = sed_df.drop_duplicates("row_id").set_index("row_id")

    perch_df = pd.DataFrame(np.vstack(perch_values), columns=target_columns)
    perch_df.insert(0, "row_id", rows)
    perch_df = perch_df.drop_duplicates("row_id").set_index("row_id")

    embedding_array = np.vstack(embedding_values).astype(np.float32)
    embedding_df = pd.DataFrame(
        embedding_array,
        columns=[f"emb_{idx}" for idx in range(embedding_array.shape[1])],
    )
    embedding_df.insert(0, "row_id", rows)
    embedding_df = embedding_df.drop_duplicates("row_id").set_index("row_id")

    row_ids = metadata["row_id"].to_numpy()
    missing = sorted(set(row_ids) - set(sed_df.index))
    if missing:
        raise ValueError(f"Missing calibration predictions: {missing[:3]}")

    sed = sed_df.loc[row_ids, target_columns].to_numpy(dtype=np.float32)
    perch = perch_df.loc[row_ids, target_columns].to_numpy(dtype=np.float32)
    embeddings = embedding_df.loc[row_ids].to_numpy(dtype=np.float32)
    return sed, perch, embeddings


def default_weight_for_class(class_idx: int) -> float:
    """Return the default Perch contribution for one target class.

    Args:
        class_idx (int): Target-column index.

    Returns:
        float: Default blend weight.
    """
    if class_idx in set(proxy_target_pos.tolist()):
        return float(CFG.proxy_weight)
    if class_idx in set(mapped_target_pos.tolist()):
        return float(CFG.perch_weight)
    return 0.0


def learn_class_weights(
    targets: np.ndarray, sed: np.ndarray, perch: np.ndarray
) -> tuple[np.ndarray, pd.DataFrame]:
    """Learn per-class Perch blend weights from soundscape labels.

    Args:
        targets (np.ndarray): Binary target matrix.
        sed (np.ndarray): SED probability matrix.
        perch (np.ndarray): Perch probability matrix.

    Returns:
        tuple[np.ndarray, pd.DataFrame]: Per-class weights and diagnostics.
    """
    weights = np.zeros(num_classes, dtype=np.float32)
    rows = []
    proxy_set = set(proxy_target_pos.tolist())
    mapped_set = set(mapped_target_pos.tolist())

    for class_idx, label in enumerate(target_columns):
        default_weight = default_weight_for_class(class_idx)
        weights[class_idx] = default_weight
        if default_weight == 0.0:
            source = "sed_only"
        elif class_idx in proxy_set:
            source = "proxy"
        elif class_idx in mapped_set:
            source = "exact"
        else:
            source = "unknown"

        y_true = targets[:, class_idx].astype(np.uint8)
        positives = int(y_true.sum())
        sed_ap = average_precision_score_binary(y_true, sed[:, class_idx])
        best_weight = default_weight
        best_ap = sed_ap

        if (
            positives >= CFG.min_positive_for_calibration
            and default_weight > 0
        ):
            for weight in CFG.calibration_grid:
                blended = (1.0 - weight) * sed[:, class_idx] + weight * perch[
                    :, class_idx
                ]
                ap = average_precision_score_binary(y_true, blended)
                if np.isnan(best_ap) or ap > best_ap:
                    best_ap = ap
                    best_weight = float(weight)
            weights[class_idx] = best_weight

        rows.append(
            {
                "primary_label": label,
                "source": source,
                "positives": positives,
                "default_weight": default_weight,
                "best_weight": float(weights[class_idx]),
                "sed_ap": sed_ap,
                "best_ap": best_ap,
            }
        )

    diagnostics = pd.DataFrame(rows)
    diagnostics.to_csv(CFG.calibration_output_path, index=False)
    return weights, diagnostics


def apply_class_blend(
    sed_probs: np.ndarray, perch_probs: np.ndarray, weights: np.ndarray
) -> np.ndarray:
    """Apply per-class SED and Perch blend weights.

    Args:
        sed_probs (np.ndarray): SED probability matrix.
        perch_probs (np.ndarray): Perch probability matrix.
        weights (np.ndarray): Per-class Perch weights.

    Returns:
        np.ndarray: Calibrated blended probabilities.
    """
    probs = (1.0 - weights[None, :]) * sed_probs + weights[None, :] * perch_probs
    return np.clip(probs, 0.0, 1.0).astype(np.float32)


def build_temporal_context(
    metadata: pd.DataFrame, base_probs: np.ndarray
) -> np.ndarray:
    """Create current, previous, and next-window probability context.

    Args:
        metadata (pd.DataFrame): Row metadata with `filename` and `end_sec`.
        base_probs (np.ndarray): Base calibrated probabilities.

    Returns:
        np.ndarray: Logit context matrix shaped `(rows, 234 * 3)`.
    """
    previous = base_probs.copy()
    following = base_probs.copy()
    ordered = metadata.reset_index().sort_values(["filename", "end_sec"])

    for _, group in ordered.groupby("filename", sort=False):
        indices = group["index"].to_numpy()
        if len(indices) <= 1:
            continue
        previous[indices[1:]] = base_probs[indices[:-1]]
        previous[indices[0]] = base_probs[indices[0]]
        following[indices[:-1]] = base_probs[indices[1:]]
        following[indices[-1]] = base_probs[indices[-1]]

    return np.concatenate(
        [logit(base_probs), logit(previous), logit(following)], axis=1
    ).astype(np.float32)


class TemporalResidualNet(nn.Module):
    """Small MLP that predicts label probabilities from temporal context."""

    def __init__(self, embedding_dim: int) -> None:
        """Initialize the residual network.

        Args:
            embedding_dim (int): Perch embedding dimension.
        """
        super().__init__()
        self.context_branch = nn.Sequential(
            nn.LayerNorm(num_classes * 3),
            nn.Linear(num_classes * 3, CFG.temporal_hidden),
            nn.ReLU(),
            nn.Dropout(CFG.temporal_dropout),
        )
        self.embedding_branch = nn.Sequential(
            nn.LayerNorm(embedding_dim),
            nn.Linear(embedding_dim, CFG.temporal_embedding_hidden),
            nn.ReLU(),
            nn.Dropout(CFG.temporal_dropout),
        )
        self.head = nn.Sequential(
            nn.Linear(
                CFG.temporal_hidden + CFG.temporal_embedding_hidden,
                CFG.temporal_hidden,
            ),
            nn.ReLU(),
            nn.Dropout(CFG.temporal_dropout),
            nn.Linear(CFG.temporal_hidden, num_classes),
        )

    def forward(
        self, context: torch.Tensor, embeddings: torch.Tensor
    ) -> torch.Tensor:
        """Predict target logits.

        Args:
            context (torch.Tensor): Temporal context features.
            embeddings (torch.Tensor): Perch embedding features.

        Returns:
            torch.Tensor: Predicted class logits.
        """
        context_features = self.context_branch(context)
        embedding_features = self.embedding_branch(embeddings)
        features = torch.cat([context_features, embedding_features], dim=1)
        return self.head(features)


def make_file_split(metadata: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Split soundscape rows into train and validation by filename.

    Args:
        metadata (pd.DataFrame): Soundscape metadata.

    Returns:
        tuple[np.ndarray, np.ndarray]: Boolean train and validation masks.
    """
    rng = np.random.default_rng(CFG.seed)
    filenames = np.array(sorted(metadata["filename"].unique()))
    shuffled = rng.permutation(filenames)
    valid_count = max(1, int(round(len(shuffled) * CFG.temporal_validation_fraction)))
    valid_files = set(shuffled[:valid_count].tolist())
    valid_mask = metadata["filename"].isin(valid_files).to_numpy()
    train_mask = ~valid_mask
    return train_mask, valid_mask


def train_temporal_residual(
    metadata: pd.DataFrame,
    targets: np.ndarray,
    base_probs: np.ndarray,
    embeddings: np.ndarray,
) -> tuple[TemporalResidualNet, pd.DataFrame]:
    """Train the lightweight temporal residual model.

    Args:
        metadata (pd.DataFrame): Soundscape metadata.
        targets (np.ndarray): Binary target matrix.
        base_probs (np.ndarray): Calibrated base probabilities.
        embeddings (np.ndarray): Perch embedding matrix.

    Returns:
        tuple[TemporalResidualNet, pd.DataFrame]: Trained model and history.
    """
    context = build_temporal_context(metadata, base_probs)
    train_mask, valid_mask = make_file_split(metadata)
    device = torch.device("cpu")

    x_context = torch.tensor(context, dtype=torch.float32, device=device)
    x_embeddings = torch.tensor(embeddings, dtype=torch.float32, device=device)
    y = torch.tensor(targets.astype(np.float32), device=device)

    positives = targets[train_mask].sum(axis=0).astype(np.float32)
    negatives = train_mask.sum() - positives
    pos_weight = negatives / np.maximum(positives, 1.0)
    pos_weight = np.clip(pos_weight, 1.0, CFG.max_pos_weight)
    pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32, device=device)

    model = TemporalResidualNet(embeddings.shape[1]).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG.temporal_learning_rate,
        weight_decay=CFG.temporal_weight_decay,
    )
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    train_indices = torch.where(torch.tensor(train_mask, device=device))[0]
    valid_indices = torch.where(torch.tensor(valid_mask, device=device))[0]
    best_state = copy.deepcopy(model.state_dict())
    best_loss = float("inf")
    stale_epochs = 0
    history = []

    for epoch in range(1, CFG.temporal_epochs + 1):
        model.train()
        order = train_indices[torch.randperm(len(train_indices))]
        train_losses = []
        for start_idx in range(0, len(order), CFG.temporal_batch_size):
            batch_idx = order[start_idx : start_idx + CFG.temporal_batch_size]
            optimizer.zero_grad(set_to_none=True)
            logits = model(x_context[batch_idx], x_embeddings[batch_idx])
            loss = loss_fn(logits, y[batch_idx])
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        model.eval()
        with torch.no_grad():
            valid_logits = model(
                x_context[valid_indices], x_embeddings[valid_indices]
            )
            valid_loss = float(
                loss_fn(valid_logits, y[valid_indices]).detach().cpu()
            )
            valid_scores = torch.sigmoid(valid_logits).cpu().numpy()

        valid_map = macro_average_precision(targets[valid_mask], valid_scores)
        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "valid_loss": valid_loss,
            "valid_map": valid_map,
        }
        history.append(row)
        print(
            f"Epoch {epoch:02d}: train_loss={row['train_loss']:.4f} "
            f"valid_loss={valid_loss:.4f} valid_map={valid_map:.4f}"
        )

        if valid_loss < best_loss - 1e-4:
            best_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= CFG.temporal_patience:
                print(f"Stopping early at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    history_df = pd.DataFrame(history)
    history_df.to_csv(CFG.temporal_history_path, index=False)
    return model, history_df


def predict_temporal_residual(
    model: TemporalResidualNet,
    metadata: pd.DataFrame,
    base_probs: np.ndarray,
    embeddings: np.ndarray,
) -> np.ndarray:
    """Predict temporal residual probabilities.

    Args:
        model (TemporalResidualNet): Trained residual model.
        metadata (pd.DataFrame): Row metadata.
        base_probs (np.ndarray): Calibrated base probabilities.
        embeddings (np.ndarray): Perch embeddings.

    Returns:
        np.ndarray: Residual model probabilities.
    """
    context = build_temporal_context(metadata, base_probs)
    device = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        logits = model(
            torch.tensor(context, dtype=torch.float32, device=device),
            torch.tensor(embeddings, dtype=torch.float32, device=device),
        )
    return torch.sigmoid(logits).cpu().numpy().astype(np.float32)


def blend_with_temporal_residual(
    base_probs: np.ndarray, residual_probs: np.ndarray
) -> np.ndarray:
    """Blend calibrated base predictions with temporal residual predictions.

    Args:
        base_probs (np.ndarray): Calibrated SED + Perch probabilities.
        residual_probs (np.ndarray): Temporal residual probabilities.

    Returns:
        np.ndarray: Final blended probabilities.
    """
    weight = CFG.temporal_residual_weight
    probs = (1.0 - weight) * base_probs + weight * residual_probs
    return np.clip(probs, 0.0, 1.0).astype(np.float32)


soundscape_metadata, soundscape_targets = build_soundscape_targets()
print("Soundscape calibration rows:", soundscape_metadata.shape)
print("Soundscape target matrix:", soundscape_targets.shape)

cal_sed, cal_perch, cal_embeddings = collect_calibration_predictions(
    soundscape_metadata
)
class_blend_weights, calibration_df = learn_class_weights(
    soundscape_targets, cal_sed, cal_perch
)
cal_base = apply_class_blend(cal_sed, cal_perch, class_blend_weights)
base_map = macro_average_precision(soundscape_targets, cal_base)
print(f"Base soundscape calibration mAP: {base_map:.4f}")

residual_model, temporal_history = train_temporal_residual(
    soundscape_metadata, soundscape_targets, cal_base, cal_embeddings
)
residual_probs = predict_temporal_residual(
    residual_model, soundscape_metadata, cal_base, cal_embeddings
)
residual_blend = blend_with_temporal_residual(cal_base, residual_probs)
residual_map = macro_average_precision(soundscape_targets, residual_blend)
print(f"Temporal residual blended mAP: {residual_map:.4f}")
print("Saved calibration diagnostics:", CFG.calibration_output_path)
print("Saved temporal history:", CFG.temporal_history_path)

display(
    calibration_df.sort_values("best_ap", ascending=False)
    .head(20)
    .reset_index(drop=True)
)
print(calibration_df["best_weight"].value_counts().sort_index())
display(temporal_history.tail())


Soundscape calibration rows: (739, 6)
Soundscape target matrix: (739, 234)
Calibration files 1/66 in 33.4s
Calibration files 10/66 in 73.0s
Calibration files 20/66 in 117.1s
Calibration files 30/66 in 159.3s
Calibration files 40/66 in 202.7s
Calibration files 50/66 in 246.8s
Calibration files 60/66 in 291.6s
Calibration files 66/66 in 317.8s
Base soundscape calibration mAP: 0.6050
Epoch 01: train_loss=0.6786 valid_loss=0.4304 valid_map=0.3747
Epoch 02: train_loss=0.3355 valid_loss=0.2461 valid_map=0.4346
Epoch 03: train_loss=0.2424 valid_loss=0.2020 valid_map=0.4439
Epoch 04: train_loss=0.1826 valid_loss=0.1855 valid_map=0.4529
Epoch 05: train_loss=0.1321 valid_loss=0.1668 valid_map=0.4814
Epoch 06: train_loss=0.1034 valid_loss=0.1562 valid_map=0.5138
Epoch 07: train_loss=0.0849 valid_loss=0.1639 valid_map=0.5174
Epoch 08: train_loss=0.0704 valid_loss=0.1758 valid_map=0.5278
Epoch 09: train_loss=0.0641 valid_loss=0.1811 valid_map=0.5198
Epoch 10: train_loss=0.0553 valid_loss=0.1883 val

,primary_label,source,positives,default_weight,best_weight,sed_ap,best_ap
0,bunibi1,exact,2,0.15,0.15,1.000000,1.000000
1,47158son14,sed_only,12,0.00,0.00,1.000000,1.000000
2,43435,exact,12,0.15,0.15,1.000000,1.000000
3,67252,exact,2,0.15,0.15,1.000000,1.000000
4,wfwduc1,exact,1,0.15,0.15,1.000000,1.000000
5,65380,exact,333,0.15,0.30,0.981794,0.995717
6,47158son06,sed_only,18,0.00,0.00,0.992063,0.992063
7,chvcon1,exact,35,0.15,0.05,0.937726,0.991470
8,hyamac1,exact,10,0.15,0.15,0.990909,0.990909
9,bufpar,exact,14,0.15,0.30,0.873509,0.987395


best_weight
0.00     25
0.05     14
0.10      2
0.15    181
0.20      1
0.25      1
0.30     10
Name: count, dtype: int64


,epoch,train_loss,valid_loss,valid_map
7,8,0.070393,0.175767,0.527751
8,9,0.064095,0.181149,0.519834
9,10,0.055316,0.188287,0.526334
10,11,0.049078,0.199143,0.530484
11,12,0.046683,0.205547,0.535956


## 6. Write Submission

During public dry runs, Kaggle may expose no `test_soundscapes`; in that case this notebook writes the untouched sample submission. During real scoring, hidden soundscapes are mounted and every expected row receives the calibrated base prediction plus the trained temporal residual blend.


In [7]:
def list_test_soundscapes() -> list[Path]:
    """List hidden test soundscapes mounted by Kaggle.

    Returns:
        list[Path]: Sorted hidden test OGG paths, or an empty list during
            public dry runs.
    """
    test_dir = CFG.data_root / "test_soundscapes"
    if not test_dir.exists():
        return []
    return sorted(test_dir.glob("*.ogg"))


def make_submission_metadata(row_ids: list[str]) -> pd.DataFrame:
    """Build row metadata for hidden soundscape predictions.

    Args:
        row_ids (list[str]): Submission row IDs.

    Returns:
        pd.DataFrame: Metadata with filename and end seconds.
    """
    rows = []
    for row_id in row_ids:
        stem, end_text = row_id.rsplit("_", 1)
        rows.append(
            {
                "row_id": row_id,
                "filename": f"{stem}.ogg",
                "end_sec": int(end_text),
            }
        )
    return pd.DataFrame(rows)


def build_submission() -> pd.DataFrame:
    """Score hidden soundscapes and write `submission.csv`.

    Returns:
        pd.DataFrame: Submission frame written to `CFG.output_path`.

    Raises:
        ValueError: If hidden scoring expects rows that were not predicted.
    """
    audio_paths = list_test_soundscapes()
    if not audio_paths:
        print("No test soundscapes found. Writing sample submission dry run.")
        submission = sample_submission.copy()
        submission.to_csv(CFG.output_path, index=False)
        return submission

    print(f"Scoring {len(audio_paths)} soundscape files")
    print(f"Exact Perch mapped columns: {len(mapped_target_pos)}")
    print(f"Default Perch blend weight: {CFG.perch_weight:.2f}")
    print(f"Default proxy blend weight: {CFG.proxy_weight:.2f}")
    print(f"Named proxy columns: {len(proxy_map)}")
    print("Using soundscape-calibrated class weights")
    print(f"Temporal residual weight: {CFG.temporal_residual_weight:.2f}")

    start = time.perf_counter()
    rows = []
    values = []

    for idx, audio_path in enumerate(audio_paths, start=1):
        row_ids, sed_probs, perch_probs, embeddings = predict_file_components(
            audio_path
        )
        base_probs = apply_class_blend(sed_probs, perch_probs, class_blend_weights)
        metadata = make_submission_metadata(row_ids)
        residual_probs = predict_temporal_residual(
            residual_model, metadata, base_probs, embeddings
        )
        probs = blend_with_temporal_residual(base_probs, residual_probs)
        rows.extend(row_ids)
        values.append(probs)
        if idx == 1 or idx % 25 == 0 or idx == len(audio_paths):
            elapsed = time.perf_counter() - start
            print(f"{idx}/{len(audio_paths)} files scored in {elapsed:.1f}s")

    pred_df = pd.DataFrame(np.vstack(values), columns=target_columns)
    pred_df.insert(0, "row_id", rows)
    pred_df = pred_df.drop_duplicates("row_id").set_index("row_id")

    missing = sorted(set(sample_submission["row_id"]) - set(pred_df.index))
    if missing:
        raise ValueError(
            f"Missing predictions for {len(missing)} row_ids. "
            f"Example: {missing[:3]}"
        )

    submission = sample_submission[["row_id"]].copy()
    submission[target_columns] = pred_df.loc[
        submission["row_id"], target_columns
    ].to_numpy()
    submission.to_csv(CFG.output_path, index=False)
    print("Saved", CFG.output_path, submission.shape)
    return submission


submission = build_submission()
display(submission.head())
print(submission.shape)


No test soundscapes found. Writing sample submission dry run.


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274


(3, 235)
